In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- CONFIGURACIÓN GLOBAL ---
# Define la ruta base de tu proyecto para guardar las imágenes
# Asume que estás ejecutando esto desde el directorio raíz del proyecto
OUTPUT_IMAGE_DIR = 'visualizaciones' 
DATA_DIR = 'data/processed'

# Crear la carpeta de visualizaciones si no existe
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
print(f"Carpeta '{OUTPUT_IMAGE_DIR}' verificada/creada.")

# Configuración de estilo de Matplotlib y Seaborn
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.1)

# Función para formatear en Guaraníes
def fmt_gs(val):
    return f"Gs. {int(val):,}".replace(",", ".")

# ----------------------------------------------------------------------------------
# PASO 1: Carga de Datos
# ----------------------------------------------------------------------------------
# Asegúrate de que estos archivos existen en la carpeta 'data/processed'
try:
    # Cargar datos históricos completos (para Series de Tiempo)
    df_historia = pd.read_csv(os.path.join(DATA_DIR, 'df_completo.csv'))
    df_historia['Fecha'] = pd.to_datetime(df_historia['Fecha'])

    # Cargar datos de clientes con clusters (para RFM)
    df_clientes = pd.read_csv(os.path.join(DATA_DIR, 'df_clientes_segmentado.csv'))

    print("✅ Datos cargados correctamente.")

except FileNotFoundError as e:
    print(f"❌ ERROR: Archivo no encontrado. Asegúrate de que 'df_completo.csv' y 'df_clientes_segmentado.csv' están en '{DATA_DIR}'")
    print("El código no puede ejecutarse sin los datos cargados/guardados previamente.")
    raise e

# ----------------------------------------------------------------------------------
# VISUALIZACIÓN 1: TENDENCIA DE VENTAS Y ESTACIONALIDAD (PPT DIAP. 3)
# Justifica el Modelo de Predicción.
# ----------------------------------------------------------------------------------
print("\nGenerando Visualización 1: Tendencia de Ventas...")

# Agregación por trimestre (más suave para PPT)
df_ventas_trimestre = df_historia.groupby(pd.Grouper(key='Fecha', freq='Q'))['Total'].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_ventas_trimestre, x='Fecha', y='Total', marker='o', color='#1f77b4', linewidth=2)
plt.title('Tendencia de Ventas Totales por Trimestre', fontsize=16)
plt.xlabel('Trimestre', fontsize=12)
plt.ylabel('Ventas Totales (Gs.)', fontsize=12)
plt.ticklabel_format(style='plain', axis='y') # Evita notación científica
plt.grid(axis='y', alpha=0.5)

# Formatear el eje Y con puntos de miles para Guaraníes (Gs.)
current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels([fmt_gs(x) for x in current_values])

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_IMAGE_DIR, '01_tendencia_ventas.png'))
plt.show()


# ----------------------------------------------------------------------------------
# VISUALIZACIÓN 2: ANÁLISIS DE PARETO (TOP 10 PRODUCTOS) (PPT DIAP. 4)
# Justifica el foco de la predicción en ítems críticos.
# ----------------------------------------------------------------------------------
print("\nGenerando Visualización 2: Análisis de Pareto (Top 10)...")

# Asumo que tienes una columna 'Producto' en df_historia para este análisis
# Si no la tienes, usa 'ID_Producto'
df_productos = df_historia.groupby('Producto')['Total'].sum().sort_values(ascending=False).reset_index()
df_productos['Porcentaje_Acumulado'] = df_productos['Total'].cumsum() / df_productos['Total'].sum()
top_10 = df_productos.head(10)

plt.figure(figsize=(12, 6))
ax1 = sns.barplot(x='Producto', y='Total', data=top_10, color='#2ca02c')
ax2 = ax1.twinx() # Eje secundario
ax2.plot(top_10['Producto'], top_10['Porcentaje_Acumulado'], color='#d62728', marker='D', linewidth=2)
ax2.axhline(0.8, color='black', linestyle='--', alpha=0.6) # Línea del 80%

plt.title('Concentración de Ingresos: Top 10 Productos', fontsize=16)
ax1.set_xlabel('Producto', fontsize=12)
ax1.set_ylabel('Ingreso Total (Gs.)', fontsize=12)
ax2.set_ylabel('Porcentaje Acumulado', fontsize=12, color='#d62728')
ax1.ticklabel_format(style='plain', axis='y') 
ax2.tick_params(axis='y', labelcolor='#d62728')

# Formatear el eje Y1 (Ingreso)
current_values = ax1.get_yticks()
ax1.set_yticklabels([fmt_gs(x) for x in current_values])

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_IMAGE_DIR, '02_pareto_productos.png'))
plt.show()

# ----------------------------------------------------------------------------------
# VISUALIZACIÓN 3: HISTOGRAMA DE GASTO (MONETARY) (PPT DIAP. 7)
# Justifica la Segmentación (Muestra el sesgo de valor).
# ----------------------------------------------------------------------------------
print("\nGenerando Visualización 3: Distribución del Gasto (Monetary)...")

plt.figure(figsize=(10, 6))
sns.histplot(df_clientes['Monetary'], bins=50, kde=True, color='#ff7f0e', edgecolor='black')

plt.title('Distribución de Gasto Total por Cliente (Monetary)', fontsize=16)
plt.xlabel('Gasto Total (Gs.)', fontsize=12)
plt.ylabel('Número de Clientes', fontsize=12)
plt.ticklabel_format(style='plain', axis='x')

# Formatear el eje X (Gasto Total)
current_values = plt.gca().get_xticks()
plt.gca().set_xticklabels([fmt_gs(x) for x in current_values], rotation=45, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_IMAGE_DIR, '03_hist_monetary.png'))
plt.show()

# ----------------------------------------------------------------------------------
# VISUALIZACIÓN 4: MAPA DE CLIENTES POR CLUSTER (K=2) (PPT DIAP. 9)
# Muestra el resultado final del modelo.
# ----------------------------------------------------------------------------------
print("\nGenerando Visualización 4: Mapa de Clientes (Clusters K=2)...")

# Ajustar las etiquetas para la presentación (Asumo Cluster 1=VIP, Cluster 0=Inactivo)
cluster_map = {1: '1: VIP / Activo', 0: '0: Inactivo / Esporádico'}
df_clientes['Segmento'] = df_clientes['Cluster'].map(cluster_map)

plt.figure(figsize=(12, 7))
# Usamos escala logarítmica en Y para manejar la dispersión de Gasto
sns.scatterplot(
    data=df_clientes, 
    x='Recency', 
    y='Monetary', 
    hue='Segmento', 
    palette=['#ff7f0e', '#1f77b4'], 
    s=70, 
    alpha=0.7
)

plt.title('Mapa de Segmentación de Clientes (Recencia vs. Gasto)', fontsize=16)
plt.xlabel('Días desde última compra (Recency)', fontsize=12)
plt.ylabel('Total Gastado (Gs. - Escala Logarítmica)', fontsize=12)
plt.yscale('log') # Escala logarítmica para ver mejor la variación del gasto
plt.legend(title='Segmento', loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_IMAGE_DIR, '04_scatter_clusters.png'))
plt.show()

print("\n✅ ¡Visualizaciones completas! Archivos .png guardados en la carpeta 'visualizaciones/'.")